# React — Forms

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> LESSON 23 showed you how to *read* an input and said plainly that reading is not
> controlling: "Making React own the value is topic 10." This is topic 10.

## LESSON 30 — Controlled inputs: `value` + `onChange`

An `<input>` normally owns the text inside it. You type, the DOM stores the characters, and
React has no idea what they are — it can listen, but it cannot answer the question *what is
in that box right now?* except by asking the DOM.

A **controlled** input is the other arrangement: React holds the text in state, and the
input is told what to display.

### The loop

This is the whole lesson, and it is a loop rather than a list:

```text
1. the input displays the current state value       value={name}
2. you type a character                             the browser fires an event
3. onChange receives that event                     event.target.value is the new text
4. the handler calls the setter                     setName(event.target.value)
5. React renders the component again                useState now returns the new text
6. value={name} puts it back on screen              step 1 again, one character longer
```

Every keystroke goes all the way round. The input is *controlled* because step 1 is where
the displayed text comes from — not because there is a `value` attribute on it.

### The API

```jsx
import { useState } from "react";

function NameBox() {
  const [name, setName] = useState("");

  function handleChange(event) {
    setName(event.target.value);
  }

  return <input value={name} onChange={handleChange} />;
}
```

React's own shape for this is the same, with the two roles commented:

```jsx
<input
  value={firstName}                              // force the input's value to match the state
  onChange={e => setFirstName(e.target.value)}   // and update the state on any edits
/>
```

### Two different values, and they are not the same thing

This is the distinction to get right, and it is the reason the loop is a loop:

| | what it is | when it exists |
|---|---|---|
| `event.target.value` | the text **the DOM has right now**, during this event | only inside the handler, while it runs |
| `value={name}` | the text **React will render next** | every render, from state |

`event.target.value` is a **reading**. It tells you what the user just did. It does not
change anything — you have to store it. `value={name}` is an **instruction**. It tells the
input what to show.

Take either one away and the arrangement breaks, in two different ways:

- **`value` with no `onChange`** — the input is frozen. It displays the state and nothing the
  user types can change it, because nothing ever updates the state. React says so out loud:
  *"You provided a `value` prop to a form field without an `onChange` handler. This will
  render a read-only field."*
- **`onChange` with no `value`** — that is LESSON 23's input. React reads every keystroke but
  the DOM still owns the text. Uncontrolled, and perfectly legal.

### `onChange` fires on every keystroke

If you know the HTML `change` event, it fires when the field loses focus. React's does not:

> Fires immediately when the input's value is changed by the user (for example, it fires on
> every keystroke). Behaves like the browser `input` event.

That is what makes the loop feel instant rather than lurching.

### Why the value has to be state

It has to survive a render and it has to be able to trigger one — LESSON 25's two reasons,
unchanged. A plain variable would be reset every render, and changing it would tell React
nothing, so the input would never be redrawn.

And because the text is now just state, **anything that can set state can set the input**.
A button elsewhere in the component that calls `setName("Ada")` changes what the box
displays, with nobody typing. That is the clearest proof of who is in charge, and the
playground makes you watch it happen.

### Start it as a string

```jsx
const [name, setName] = useState("");   // not useState()
```

React is explicit: the value you pass to a controlled component *should not be `undefined` or
`null`*. `useState()` gives `undefined`, which makes the input uncontrolled on the first
render and controlled the moment you type — and React complains about exactly that switch:

> A component is changing an uncontrolled input to be controlled.

An empty string is the empty value. Use it.

### Key Notes

- Controlled means the displayed text **comes from state**: `value={name}` plus an
  `onChange` that updates `name`. Both props, or it is not controlled.
- `event.target.value` is what the DOM has during the event; `value={name}` is what React
  will render next. Reading is not controlling.
- React's `onChange` fires on every keystroke, unlike the HTML `change` event.
- Initialise the state to `""`, never `undefined`.

### Example

**In the playground.** There is no cell for this one. The whole point is the order in which
real things happen — event, then state, then render — and a plain-JavaScript imitation would
teach you my imitation instead of React.

Point `playground/src/App.jsx` at `./experiments/10-controlled.jsx`, run it, and open the
console. The experiment has a controlled input, LESSON 23's uncontrolled one for contrast,
and a button that sets the state without anyone typing.

Type one character in the controlled input and read the console. You will see this order:

```text
   onChange — event.target.value is "A"
   render — name is "A"
```

The handler runs first and reads the DOM. The render happens afterwards, and only then does
`value={name}` put the character on screen. It is fast enough to feel like nothing happened
— but that is the loop, once per keystroke.

### Exercise

**In the playground**, in `10-controlled.jsx`. You are going to build a controlled input from
nothing, then convert one.

**Part 1 — build one.** Add a second piece of state called `city`, starting as an empty
string, and a second controlled input for it. Do not copy the first input — write the four
pieces yourself and check each one: the state, the `value` prop, the handler, and the setter
call. Confirm that typing in `city` leaves `name` alone.

**Part 2 — convert one.** The `uncontrolled` input has only an `onChange`. Make it
controlled, using `name` as its value, so that it and the first input always show the same
text. Type in either and watch both.

**Part 3 — prove who owns it.** Press **Set both to Ada** and write down, in one sentence,
why the controlled inputs change and the uncontrolled one — before you converted it — did
not.

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

Three inputs, three different mistakes. For each one, say **what the user sees** and **which
part of the loop is broken** — then write the corrected line.

```jsx
// 1
<input value="Sam" />

// 2
const [name, setName] = useState("");
<input value={name} onChange={setName} />

// 3
const [name, setName] = useState();
<input value={name} onChange={(e) => setName(e.target.value)} />
```

Number 2 is the one worth thinking hardest about: nothing throws, and React logs no warning
at all. Work out what actually ends up in `name`, and what the input therefore displays.

In [ ]:
// Your code here

## LESSON 31 — One state object for many fields

LESSON 30 gave one input its own state variable. Two fields means two of them, and two
handlers that differ by one word:

```jsx
const [firstName, setFirstName] = useState("");
const [email, setEmail] = useState("");
```

Five fields means five of each. You have met this shape before — LESSON 24 had three toolbar
buttons wanting three nearly identical handlers, and the answer was to let the element carry
its own identity. Forms get the same treatment, and the identity is already built in.

### One object instead of many variables

```jsx
const [values, setValues] = useState({ firstName: "", email: "" });
```

Each input reads its own field out of it:

```jsx
<input name="firstName" value={values.firstName} onChange={handleChange} />
<input name="email"     value={values.email}     onChange={handleChange} />
```

Still controlled, exactly as in LESSON 30 — the displayed text still comes from state. All
that changed is where the state lives.

### The input already knows which field it is

`name` is a plain HTML attribute, and React passes it through to the DOM. Inside the handler,
`event.target.name` reads it back. This is LESSON 24's `data-action` trick, except you do not
need a `data-*` attribute: `name` is already there, and identifying a field is what it is for.

> LESSON 24 told you to prefer `currentTarget` over `target`, because a click can land on
> something *inside* the element you wired up. An input has nothing inside it to receive the
> event, so for its own `onChange` the two are the same element — measured in the experiment:
> `target === currentTarget? true`. React's docs use `target` here, and so will we.

### One handler for all of them

```jsx
function handleChange(event) {
  const { name, value } = event.target;
  setValues({ ...values, [name]: value });
}
```

React's docs write it in one expression:

```js
setPerson({
  ...person,
  [e.target.name]: e.target.value
});
```

> …where each input has a `name` attribute matching the property to update.

Nothing in that line is new to you. Three things you already have, doing one job:

| | |
|---|---|
| `{ ...values }` | copies every field — LESSON 27 |
| `[name]` | a computed key: the variable supplies the property name — the JavaScript course |
| the later key wins | so only the named field is replaced, and the rest survive |

That last column is the whole point. One handler updates whichever field was typed in, and
leaves every other field exactly as it was.

### The one rule to get right

**The `name` on the input must match the property in state, character for character.** Nothing
enforces it and nothing warns you. Get it wrong and you do not get an error — you get a *new*
property quietly added to the object, while the field on screen never changes, because
`value={values.firstName}` is still reading the property nobody is writing to.

### Key Notes

- Related fields belong in one state object; one handler can then serve all of them.
- `name` on the input identifies the field, and `event.target.name` reads it back.
- `{ ...values, [name]: value }` copies everything, then replaces one field — the other
  fields survive because you copied them, not because React merged anything.
- The `name` must match the state property exactly. A mismatch fails silently.

### Example

**Runnable — plain JS.** The update itself is ordinary JavaScript and worth having on its own,
away from React. Watch the second field across every call: it is never named, and it is never
lost.

In [ ]:
function l31updateField(values, name, value) {
  return { ...values, [name]: value };
}

const l31start = { firstName: "", email: "" };

const l31a = l31updateField(l31start, "firstName", "Ada");
const l31b = l31updateField(l31a, "email", "ada@example.com");

console.log("start:", JSON.stringify(l31start));
console.log("a:    ", JSON.stringify(l31a));
console.log("b:    ", JSON.stringify(l31b));

// The original was never touched, and each step is a new object.
console.log("start untouched?", JSON.stringify(l31start) === '{"firstName":"","email":""}');
console.log("new object each time?", !Object.is(l31start, l31a) && !Object.is(l31a, l31b));

// And the silent failure: a name that does not match any property.
const l31typo = l31updateField(l31b, "e-mail", "typo@example.com");
console.log("typo:  ", JSON.stringify(l31typo));
console.log("email unchanged?", l31typo.email === "ada@example.com");

The last two lines are the one to remember. Nothing threw, nothing warned, and the object now
carries a property no input will ever display.

### Exercise

**Part 1 — in the notebook.**

1. Write `l31applyEdits(values, edits)` which takes a values object and an **array** of
   `[name, value]` pairs, and returns the final values object after applying them in order.
   Use `l31updateField` rather than rewriting the spread. Start from
   `{ firstName: "", email: "", city: "" }` and apply three edits, one of them overwriting an
   earlier field.
2. Prove two things and log both: the starting object is unchanged, and the result has
   exactly three properties (no stray ones).
3. In a comment: your form has an input with `name="firstname"` while state has `firstName`.
   Describe precisely what the user sees when they type in it, and why nothing reports a
   problem.

**Part 2 — in the playground**, in `11-fields.jsx`. Add a **third** field, `city`, to the same
state object:

1. Add it to the object passed to `useState`.
2. Add its input, with the right `name` and `value`.
3. Write no new handler — `handleChange` must already work.

Type in all three and watch the logged object. Then deliberately misspell the new input's
`name`, type into it, and read the state line: the point is what *does not* happen.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Four handlers, four different outcomes. For each, say what ends up in state after typing `A`
into the `firstName` input, starting from `{ firstName: "", email: "ada@example.com" }`.

```jsx
// 1
setValues({ [name]: value });

// 2
values[name] = value;
setValues(values);

// 3
setValues({ ...values, name: value });

// 4
setValues({ ...values, [name]: value });
```

Then answer: which of the three broken ones is hardest to notice in a real form, and why?

In [ ]:
// Your code here

## LESSON 32 — Submitting: `onSubmit`, `preventDefault`, and resetting

You now have fields that hold their values in state. What is missing is the moment the user
says *done* — and a `<form>` element, which you have not needed until now.

### Why a `<form>` at all

You could put an `onClick` on a button and skip the form element entirely. You would lose
things the browser gives you for free, and the useful one is this: **pressing Enter in a text
field submits the form.** Users expect that, and reimplementing it by hand means listening for
keys. Wrap the fields in a `<form>` and it works.

```jsx
<form onSubmit={handleSubmit}>
  <input name="firstName" value={values.firstName} onChange={handleChange} />
  <button type="submit">Add</button>
</form>
```

The handler goes on the **form**, not on the button — because the form is what gets submitted,
whether that came from a click or from the Enter key.

### The browser wants to submit it itself

A form is older than React and has its own behaviour. React's docs are direct about it:

> By default, the browser sends the form data to the current URL and refreshes the page, so
> call `e.preventDefault()` to override that behavior.

This is not theoretical. With the line commented out, submitting the playground form does
exactly that — measured:

```text
before:  http://localhost:5392/
after:   http://localhost:5392/?firstName=Ada&email=ada%40x.dev
```

Three things happened there. The browser collected the fields — **using their `name`
attributes as the keys**, the same names LESSON 31 used — put them in the URL, and *navigated*.
The page reloaded from scratch, so every piece of state was destroyed: the list of submitted
entries was empty again, and so were the inputs.

### `event.preventDefault()`

```jsx
function handleSubmit(event) {
  event.preventDefault();
  // now it is your form
}
```

You met this in LESSON 23 on a link. Same method, same meaning:

> Prevents the default browser action for the event.

**Be precise about what it does and does not do.** It cancels the *browser's* built-in
response to the event. It does not:

- stop your handler — your handler is what called it, and the rest of it still runs;
- stop React — React is not involved in the default action at all, and rendering continues
  normally;
- stop the event bubbling — that is `stopPropagation()`, a different method for a different
  job (LESSON 23).

### Reading the values on submit

There is nothing to collect. The values are already in state, because every keystroke put
them there:

```jsx
function handleSubmit(event) {
  event.preventDefault();
  addEntry({ ...values });      // values is right here
}
```

> React's own `onSubmit` example reads the fields with `FormData` instead — because the form
> in that example is **uncontrolled**, so the values only exist in the DOM. With controlled
> inputs you have already done that work, once per keystroke.

One thing to keep from LESSON 26: inside the handler, `values` is *this render's* values.
That is exactly what you want — it is what the user saw when they pressed the button.

### Resetting

A reset is not a special operation. The form displays state, so putting the state back to its
starting value empties the boxes:

```jsx
const emptyForm = { firstName: "", email: "" };   // outside the component

function handleSubmit(event) {
  event.preventDefault();
  addEntry({ ...values });
  setValues(emptyForm);          // the inputs are now empty
}
```

Declaring `emptyForm` **outside** the component means there is one of it, and both
`useState(emptyForm)` and the reset use the same starting point. Nothing about this is new —
it is LESSON 30's loop running with a value you chose instead of one the user typed.

> **A note for when you read the docs.** React 19 also lets you pass a *function* to a form's
> `action` prop, which handles submission without `preventDefault`. That is a different and
> more capable mechanism — it is topic 25, after you have met transitions — and `onSubmit`
> remains correct and widely used. Learn this one first.

### Key Notes

- Put `onSubmit` on the `<form>`, not on the button; the form is what submits, and Enter
  submits it too.
- The browser's default is to serialise the fields into the URL and reload the page.
  `event.preventDefault()` cancels that.
- `preventDefault()` stops the *browser's* default action — not your handler, not React, and
  not bubbling.
- The values are already in state; resetting means setting the state back to its starting
  object.

### Example

**In the playground.** There is no cell for this one either — a navigation and a page reload
are browser behaviour, and the address bar is the evidence.

Point `playground/src/App.jsx` at `./experiments/12-submit.jsx` and run it. Fill both fields
and press **Add**. The entry joins the list, the boxes empty, and the URL does not change.

### Exercise

**In the playground**, in `12-submit.jsx`.

**Part 1 — see the default.** Comment out the `event.preventDefault()` line. Fill the fields
and submit. Write down, in comments: the URL before, the URL after, where the `firstName` and
`email` names in that URL came from, and what happened to the list of submitted entries.
Then put the line back.

**Part 2 — build one yourself.** Add a third field, `city`, and then check your form does all
five of these. Tick them off one at a time:

1. the input is controlled
2. submission goes through `onSubmit` on the `<form>`
3. the browser's navigation is prevented
4. the entry added to the list uses the **current** state
5. the form resets after a successful submit

**Part 3 — Enter.** Click into a text field and press Enter without touching the button. It
submits. Say in one sentence why the handler still runs, given that no button was clicked.

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

Four submit handlers. For each, say what the user experiences.

```jsx
// 1
function handleSubmit(event) {
  addEntry({ ...values });
  setValues(emptyForm);
}

// 2 — the handler is on the button instead of the form
<form>
  <input name="firstName" value={values.firstName} onChange={handleChange} />
  <button type="submit" onClick={handleSubmit}>Add</button>
</form>

// 3
function handleSubmit(event) {
  event.stopPropagation();
  addEntry({ ...values });
}

// 4
function handleSubmit(event) {
  event.preventDefault();
  setValues(emptyForm);
  addEntry({ ...values });
}
```

Number 4 looks wrong — the state is reset on the line *before* the entry is added. LESSON 26
already told you what actually happens. Say what ends up in the list, and why.

In [ ]:
// Your code here

## LESSON 33 — Validation as pure logic

A form that accepts anything is not finished. But validation has a way of getting tangled
into the component until you cannot test it, reuse it or read it — so this lesson is as much
about *where the code goes* as about the checks themselves.

### Two jobs, not one

| | job | where it lives |
|---|---|---|
| **validating** | given these values, what is wrong? | a plain function, outside the component |
| **displaying** | show the user what is wrong | the component, from state |

Keeping them apart is the whole lesson. The function decides; React shows.

### Make the validator pure

React's definition of a pure function is the one to hold on to:

> - **It minds its own business.** It does not change any objects or variables that existed
>   before it was called.
> - **Same inputs, same output.** Given the same inputs, a pure function should always return
>   the same result.

A validator that obeys those two is trivial to reason about and trivial to check: hand it
values, look at what comes back.

### The contract

```js
validate(values) -> errors
```

- `errors` has **one entry per invalid field**, keyed by the field name:
  `{ firstName: "First name is required." }`
- valid input returns **`{}`** — an empty object, not `null` and not `true`
- the values passed in are **never changed**

Checking validity is then one expression:

```js
const errors = validate(values);
const isValid = Object.keys(errors).length === 0;
```

Why an object rather than a boolean or a list? Because the component has to put each message
next to the right input, and a key per field is exactly that lookup. `errors.email` is either
a message or `undefined`, which is all the conditional rendering from LESSON 18 needs.

### The validator

```js
const emptyForm = { firstName: "", email: "" };

export function validate(values) {
  const errors = {};

  const firstName = values.firstName.trim();
  if (firstName === "") {
    errors.firstName = "First name is required.";
  } else if (firstName.length < 2) {
    errors.firstName = "First name must be at least 2 characters.";
  }

  const email = values.email.trim();
  if (email === "") {
    errors.email = "Email is required.";
  } else if (!email.includes("@") || email.startsWith("@") || email.endsWith("@")) {
    errors.email = "Email must look like name@example.com.";
  }

  return errors;
}
```

Note `.trim()` is used on a **copy** — `values.firstName.trim()` returns a new string and
leaves `values` untouched. And `errors` is built fresh inside the function, so filling it in
is LESSON 27's *local mutation*: the object did not exist before the call, so nothing else can
be affected by it.

> **On checking emails.** This is a shape check for helpful feedback, not proof an address
> exists. Do not go looking for the perfect email regular expression — it is a famous rabbit
> hole, and the only real check is sending a message to it.

### The component's half

```jsx
const [errors, setErrors] = useState({});

function handleSubmit(event) {
  event.preventDefault();

  const found = validate(values);
  setErrors(found);

  if (Object.keys(found).length > 0) {
    return;                      // stop: nothing is added, nothing is reset
  }

  addEntry({ ...values });
  setValues(emptyForm);
}
```

Three things worth naming:

- `validate` is called **on submit** — that is enough for now, and it means the user is not
  told they are wrong while still typing their first letter.
- The result goes into state, because rendering needs it and only state survives a render
  (LESSON 25).
- The early `return` is LESSON 18's early return doing ordinary work: invalid means stop.

And the display, next to each field:

```jsx
<input name="email" value={values.email} onChange={handleChange} />
{errors.email && <span className="error">{errors.email}</span>}
```

`errors.email` is `undefined` when the field is fine, so nothing renders — the `&&` pattern
from LESSON 18, with a string rather than a number, so no `0` trap here.

### Key Notes

- Validation is a **pure function** of the values: same input, same output, and it changes
  nothing it was given.
- The contract is `validate(values) -> errors`, one entry per invalid field, `{}` when valid.
- Keep it outside the component. The component decides *when* to call it and *what to show*.
- Deciding and displaying are different jobs — the validator never touches state.

### Example

**Runnable — plain JS.** This is the validator itself, with nothing React anywhere near it.
That is the point: it is ordinary JavaScript you can call, test and reuse.

In [ ]:
function l33validate(values) {
  const errors = {};

  const firstName = values.firstName.trim();
  if (firstName === "") {
    errors.firstName = "First name is required.";
  } else if (firstName.length < 2) {
    errors.firstName = "First name must be at least 2 characters.";
  }

  const email = values.email.trim();
  if (email === "") {
    errors.email = "Email is required.";
  } else if (!email.includes("@") || email.startsWith("@") || email.endsWith("@")) {
    errors.email = "Email must look like name@example.com.";
  }

  return errors;
}

const l33good = { firstName: "Ada", email: "ada@example.com" };
const l33empty = { firstName: "", email: "" };
const l33short = { firstName: "A", email: "nope" };

console.log("valid   ->", JSON.stringify(l33validate(l33good)));
console.log("empty   ->", JSON.stringify(l33validate(l33empty)));
console.log("short   ->", JSON.stringify(l33validate(l33short)));

// Valid means an EMPTY object, so this is the test:
console.log("is valid?", Object.keys(l33validate(l33good)).length === 0);

// Same input, same output - run it twice and compare.
console.log("deterministic?", JSON.stringify(l33validate(l33short)) === JSON.stringify(l33validate(l33short)));

// And it minds its own business: the values it was handed are untouched.
const l33before = JSON.stringify(l33short);
l33validate(l33short);
console.log("values unchanged?", JSON.stringify(l33short) === l33before);

Those last three lines are not decoration — they are the three properties that make a pure
function worth having, checked in three lines each. You cannot do that to a validator tangled
into a component.

### Exercise

**Part 1 — in the notebook.** Extend the contract without breaking it.

1. Write `l33validateWithAge(values)` which does everything `l33validate` does **and** checks a
   new `age` field: required, and it must be a number between 18 and 120. The values arrive as
   strings, as they always do from an input.
2. Prove all four properties of the contract, and log each proof:
   - a fully valid object returns `{}`
   - an object wrong in all three fields returns exactly three keys
   - calling it twice on the same input gives the same result
   - the values object is unchanged afterwards
3. In a comment: why does the validator return messages rather than calling something to show
   them?

**Part 2 — in the playground**, in `13-validation.jsx`. Add the `age` field to the form:

1. Add it to `emptyForm` and give it a controlled input (LESSON 30–31 — no new handler).
2. Swap in your new validator.
3. Submit with it empty, with `abc`, with `9`, and with `30`, and check the right message
   appears next to the right field each time.

Notice what you did **not** have to change: `handleSubmit` still just calls `validate`, stores
the result, and stops when there is anything in it.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Four validators. Each one breaks the contract or the purity rule. For each, say which rule it
breaks and what goes wrong in practice.

```js
// 1
function validate(values) {
  values.firstName = values.firstName.trim();
  if (values.firstName === "") return { firstName: "Required." };
  return {};
}

// 2
function validate(values) {
  return values.firstName !== "" && values.email.includes("@");
}

// 3
function validate(values) {
  const errors = {};
  if (new Date().getHours() > 17) errors.form = "Submissions close at 5pm.";
  return errors;
}

// 4
function validate(values, setErrors) {
  const errors = {};
  if (values.email === "") errors.email = "Required.";
  setErrors(errors);
}
```

Then answer: number 3 enforces a real business rule that a form might genuinely need. Where
should that check live instead, and why not in `validate`?

In [ ]:
// Your code here